In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [7]:
books = pd.read_csv(
    'books.csv',        # or full path like '/mnt/user-data/uploads/books.csv'
    sep=';',
    encoding='latin-1',
    on_bad_lines='skip',
    dtype={'Year-Of-Publication': str} 
)

In [8]:
ratings = pd.read_csv(
    'ratings.csv',
    sep=';',
    encoding='latin-1',
    on_bad_lines='skip'
)

In [9]:
users = pd.read_csv(
    'users.csv',
    sep=';',
    encoding='latin-1',
    on_bad_lines='skip'
)

In [10]:
print("=" * 60)
print("STEP 1: DATA LOADING COMPLETE")
print("=" * 60)
print(f"Books   : {books.shape[0]:,} rows × {books.shape[1]} cols")
print(f"Ratings : {ratings.shape[0]:,} rows × {ratings.shape[1]} cols")
print(f"Users   : {users.shape[0]:,} rows × {users.shape[1]} cols")
print()


STEP 1: DATA LOADING COMPLETE
Books   : 271,360 rows × 8 cols
Ratings : 1,149,780 rows × 3 cols
Users   : 278,858 rows × 3 cols



In [11]:
print("Books columns :", list(books.columns))
print("Ratings columns:", list(ratings.columns))
print("Users columns  :", list(users.columns))
 

Books columns : ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']
Ratings columns: ['User-ID', 'ISBN', 'Book-Rating']
Users columns  : ['User-ID', 'Location', 'Age']


STEP 2: EXPLORATORY DATA ANALYSIS

In [12]:
# Rating distribution
print("\nRating Value Counts:")
print(ratings['Book-Rating'].value_counts().sort_index())
# Note: Rating=0 means 'implicit' (viewed but not rated) — we'll filter these out


Rating Value Counts:
Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64


In [13]:
# How many unique users/books interact?
print(f"\nUnique users who rated: {ratings['User-ID'].nunique():,}")
print(f"Unique books rated     : {ratings['ISBN'].nunique():,}")


Unique users who rated: 105,283
Unique books rated     : 340,556


In [14]:
# Sparsity
n_users = ratings['User-ID'].nunique()
n_books = ratings['ISBN'].nunique()
n_ratings = len(ratings)
sparsity = 1 - (n_ratings / (n_users * n_books))
print(f"\nMatrix Sparsity: {sparsity:.4%}")
# Very sparse matrices are common in recommendation systems — CF handles this.


Matrix Sparsity: 99.9968%


In [15]:
# Books missing metadata
print(f"\nMissing in books:")
print(books.isnull().sum())


Missing in books:
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64


 STEP 3: DATA CLEANING & PREPROCESSING

In [ ]:
# 3a. Drop implicit ratings (0 = no explicit rating) 
explicit_ratings = ratings[ratings['Book-Rating'] > 0].copy()
print(f"Explicit ratings (score 1–10): {len(explicit_ratings):,}")
print(f"Dropped implicit (score=0)   : {len(ratings) - len(explicit_ratings):,}")

Explicit ratings (score 1–10): 433,671
Dropped implicit (score=0)   : 716,109


In [ ]:
# 3b. Filter cold-start: keep active users & popular books 
# Users with at least 5 ratings
user_counts = explicit_ratings['User-ID'].value_counts()
active_users = user_counts[user_counts >= 5].index
explicit_ratings = explicit_ratings[explicit_ratings['User-ID'].isin(active_users)]

In [18]:
# Books with at least 5 ratings
book_counts = explicit_ratings['ISBN'].value_counts()
popular_books = book_counts[book_counts >= 5].index
explicit_ratings = explicit_ratings[explicit_ratings['ISBN'].isin(popular_books)]

In [19]:
print(f"\nAfter cold-start filtering:")
print(f"  Ratings : {len(explicit_ratings):,}")
print(f"  Users   : {explicit_ratings['User-ID'].nunique():,}")
print(f"  Books   : {explicit_ratings['ISBN'].nunique():,}")


After cold-start filtering:
  Ratings : 141,081
  Users   : 13,030
  Books   : 11,234


In [ ]:
# 3c. Clean books metadata
books['Book-Title']  = books['Book-Title'].fillna('Unknown Title').str.strip()
books['Book-Author'] = books['Book-Author'].fillna('Unknown Author').str.strip()
books['Publisher']   = books['Publisher'].fillna('Unknown Publisher').str.strip()

In [22]:
# Fix year: keep only valid years (1800–2024)
books['Year-Of-Publication'] = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')
books['Year-Of-Publication'] = books['Year-Of-Publication'].where(
    books['Year-Of-Publication'].between(1800, 2024)
)

In [ ]:
# 3d. Merge ratings with book metadata 
df = explicit_ratings.merge(
    books[['ISBN', 'Book-Title', 'Book-Author', 'Publisher', 'Year-Of-Publication']],
    on='ISBN',
    how='inner'
)
print(f"\nMerged dataframe shape: {df.shape}")
print(df.head(3).to_string())


Merged dataframe shape: (137214, 7)
   User-ID        ISBN  Book-Rating                Book-Title    Book-Author    Publisher  Year-Of-Publication
0   276747  0060517794            9  Little Altars Everywhere  Rebecca Wells  HarperTorch               2003.0
1   278843  0060517794            7  Little Altars Everywhere  Rebecca Wells  HarperTorch               2003.0
2     4017  0060517794           10  Little Altars Everywhere  Rebecca Wells  HarperTorch               2003.0


STEP 4: CONTENT-BASED FILTERING

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 4a. Build a content "soup" for each book
# We combine Title + Author + Publisher into a single text field
# This forms the "content profile" for each book
book_meta = books[books['ISBN'].isin(df['ISBN'].unique())].copy()
book_meta = book_meta.drop_duplicates(subset='ISBN').reset_index(drop=True)
book_meta['content'] = (
    book_meta['Book-Title'].fillna('') + ' ' +
    book_meta['Book-Author'].fillna('') + ' ' +
    book_meta['Publisher'].fillna('')
).str.lower()

In [26]:

print(f"Books with content profiles: {len(book_meta):,}")
print("\nSample content soup:")
print(book_meta[['Book-Title', 'content']].head(3).to_string())

Books with content profiles: 10,811

Sample content soup:
                                                                                           Book-Title                                                                                                                                   content
0                                                                                        Clara Callan                                                                                   clara callan richard bruce wright harperflamingo canada
1  Flu: The Story of the Great Influenza Pandemic of 1918 and the Search for the Virus That Caused It  flu: the story of the great influenza pandemic of 1918 and the search for the virus that caused it gina bari kolata farrar straus giroux
2                                                                              The Kitchen God's Wife                                                                                           the kitchen god's wife amy tan

In [ ]:
# 4b. TF-IDF Vectorization
# TF-IDF: Term Frequency-Inverse Document Frequency
# - Words common across all books get low weight (e.g., "the", "and")
# - Words unique to a specific book get high weight (e.g., author name)
# max_features=5000 keeps only the 5000 most informative terms
 
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',   # remove common English words
    ngram_range=(1, 2)      # use single words AND two-word phrases
)

In [28]:
tfidf_matrix = tfidf.fit_transform(book_meta['content'])
print(f"\nTF-IDF matrix shape: {tfidf_matrix.shape}")
# Shape = (n_books, n_features)


TF-IDF matrix shape: (10811, 5000)


In [ ]:
# 4c. Content-Based Recommender Function 
# We build an index so we can look up a book by title
book_indices = pd.Series(book_meta.index, index=book_meta['Book-Title'].str.lower())

In [ ]:
def content_based_recommend(title, n=10):
    """
    Given a book title, return top-N similar books using cosine similarity
    on TF-IDF vectors (title + author + publisher)
    """
    title_lower = title.lower()
 
    # Find the book in our index
    if title_lower not in book_indices:
        # Fuzzy match: try partial substring
        matches = [t for t in book_indices.index if title_lower in t]
        if not matches:
            return pd.DataFrame({'Error': [f"'{title}' not found in dataset"]})
        title_lower = matches[0]
 
    idx = book_indices[title_lower]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]   # handle duplicates
 
    # Compute cosine similarity of this book against all others
    # cosine_similarity returns values between 0 (no match) and 1 (identical)
    book_vector = tfidf_matrix[idx]
    sim_scores  = cosine_similarity(book_vector, tfidf_matrix).flatten()
 
    # Get top-N (excluding the query book itself)
    sim_indices = np.argsort(sim_scores)[::-1][1:n+1]
 
    results = book_meta.iloc[sim_indices][['Book-Title', 'Book-Author', 'Publisher']].copy()
    results['similarity_score'] = sim_scores[sim_indices].round(4)
    results.reset_index(drop=True, inplace=True)
    return results

In [33]:
print("\nContent-Based Recommendations for 'Classical Mythology':")
print(content_based_recommend('Classical Mythology', n=5).to_string())


Content-Based Recommendations for 'Classical Mythology':
                                        Error
0  'Classical Mythology' not found in dataset


In [34]:
'Classical Mythology' in book_meta['Book-Title'].values

False

STEP 5: COLLABORATIVE FILTERING (SVD)

In [37]:
from scipy.sparse import csr_matrix
from sklearn.utils.extmath import randomized_svd
from sklearn.model_selection import train_test_split as sklearn_train_test_split
from collections import defaultdict

In [ ]:
# 5a. Encode user/book IDs as contiguous integers 
all_users = df['User-ID'].unique()
all_isbns = df['ISBN'].unique()

In [39]:
user2idx = {u: i for i, u in enumerate(all_users)}
isbn2idx = {b: i for i, b in enumerate(all_isbns)}
idx2user = {i: u for u, i in user2idx.items()}
idx2isbn = {i: b for b, i in isbn2idx.items()}

In [40]:
n_u = len(all_users)
n_b = len(all_isbns)

In [41]:
df['u_idx'] = df['User-ID'].map(user2idx)
df['b_idx'] = df['ISBN'].map(isbn2idx)


In [42]:
print(f"User-Item matrix size: {n_u:,} users × {n_b:,} books")

User-Item matrix size: 12,873 users × 10,811 books


In [ ]:
# 5b. Train / Test split (80 / 20) 
train_df, test_df = sklearn_train_test_split(df, test_size=0.2, random_state=42)
print(f"Training samples : {len(train_df):,}")
print(f"Test samples     : {len(test_df):,}")

Training samples : 109,771
Test samples     : 27,443


In [ ]:
# 5c. Build sparse rating matrix on TRAIN set
row    = train_df['u_idx'].values
col    = train_df['b_idx'].values
data   = train_df['Book-Rating'].values.astype(float)
R_sparse = csr_matrix((data, (row, col)), shape=(n_u, n_b))

In [ ]:
# 5d. Mean-centre per user (subtract each user's average rating) 
#   This removes individual rating-scale bias
#   (some users always rate high, others always rate low)
user_means = np.zeros(n_u)
for u in range(n_u):
    user_row = R_sparse.getrow(u)
    if user_row.nnz > 0:
        user_means[u] = user_row.data.mean()

In [46]:
# Build mean-centred matrix (dense, but only non-zero entries)
R_centred = R_sparse.copy().astype(float)
for u in range(n_u):
    if R_centred.getrow(u).nnz > 0:
        R_centred[u] = R_centred.getrow(u)
        # Subtract mean from non-zero entries only
        R_centred.data[R_centred.indptr[u]:R_centred.indptr[u+1]] -= user_means[u]
 

In [ ]:
# 5e. Truncated SVD 
#   n_components = number of latent factors (taste dimensions)
#   Higher k → more expressive model, but slower & risks overfitting
N_FACTORS = 50   # good balance for book-crossing dataset size
 
print(f"\nRunning Truncated SVD with {N_FACTORS} latent factors …")
U, sigma, Vt = randomized_svd(
    R_centred,
    n_components=N_FACTORS,
    random_state=42
)


Running Truncated SVD with 50 latent factors …


In [48]:
# U     : (n_users, k)  — user embeddings
# sigma : (k,)          — singular values (importance of each factor)
# Vt    : (k, n_books)  — book embeddings
 
# Pre-compute U × diag(sigma) for fast prediction
U_sigma = U * sigma          # (n_users, k) — each row is a user's weighted vector
print("SVD factorization complete.")
print(f"  U      : {U.shape}")
print(f"  sigma  : {sigma.shape}")
print(f"  Vt     : {Vt.shape}")

SVD factorization complete.
  U      : (12873, 50)
  sigma  : (50,)
  Vt     : (50, 10811)


In [ ]:
# 5f. Prediction helper 
def predict_rating(u_idx, b_idx):
    """
    Predict the rating user u_idx would give book b_idx.
    Returns: float in approximately [1, 10]
    """
    pred = user_means[u_idx] + U_sigma[u_idx] @ Vt[:, b_idx]
    return float(np.clip(pred, 1.0, 10.0))   # clip to valid rating range

In [ ]:

# 5g. CF Recommender Function 
def cf_recommend(user_id, n=10):
    """
    For a given user, predict ratings for all unread books and return top-N.
    """
    if user_id not in user2idx:
        return pd.DataFrame({'Error': [f"User {user_id} not in training data"]})
 
    u_idx = user2idx[user_id]
 
    # Books this user has already rated (in full dataset)
    rated_isbns = set(df[df['User-ID'] == user_id]['ISBN'].values)
 
    # Vectorised prediction for ALL books at once (fast)
    # predicted = user_mean + U_sigma[u] · Vt   →  shape (n_books,)
    all_preds = user_means[u_idx] + U_sigma[u_idx] @ Vt
    all_preds = np.clip(all_preds, 1.0, 10.0)
 
    # Build result, filter out already-rated books
    cand_df = book_meta.copy()
    cand_df['b_idx'] = cand_df['ISBN'].map(isbn2idx)
    cand_df = cand_df.dropna(subset=['b_idx'])
    cand_df['b_idx'] = cand_df['b_idx'].astype(int)
    cand_df = cand_df[~cand_df['ISBN'].isin(rated_isbns)]
    cand_df['predicted_rating'] = all_preds[cand_df['b_idx'].values].round(3)
 
    results = (
        cand_df[['ISBN', 'Book-Title', 'Book-Author', 'predicted_rating']]
        .sort_values('predicted_rating', ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    return results

In [51]:
# Quick test
sample_user = df['User-ID'].value_counts().index[0]
print(f"\nCF Recommendations for User {sample_user}:")
print(cf_recommend(sample_user, n=5).to_string())


CF Recommendations for User 11676:
         ISBN                            Book-Title         Book-Author  predicted_rating
0  0671027360                   Angels &amp; Demons           Dan Brown             7.593
1  0451183665                        A Case of Need    Michael Crichton             7.586
2  0671524313   The Girlfriends' Guide to Pregnancy        Vicki Iovine             7.584
3  0786889020  Don't Stand Too Close to a Naked Man           Tim Allen             7.583
4  0060930535         The Poisonwood Bible: A Novel  Barbara Kingsolver             7.582


STEP 6: HYBRID MODEL

In [ ]:
def hybrid_recommend(user_id, seed_title, n=10, alpha=0.6):
    """
    Hybrid recommender combining CF and Content-Based filtering

    Parameters
    user_id    : int   — target user
    seed_title : str   — a book the user liked (used for content signal)
    n          : int   — number of recommendations
    alpha      : float — weight for CF score (1-alpha goes to content score)
    """

    from difflib import get_close_matches
    import numpy as np
    import pandas as pd
    from sklearn.metrics.pairwise import cosine_similarity

    # 1. Check user 
    if user_id not in user2idx:
        print(f"User {user_id} not in training data.")
        return None

    # 2. Find seed book (robust matching)
    seed_lower = seed_title.lower()
    seed_idx = None

    if seed_lower in book_indices:
        seed_idx = book_indices[seed_lower]
    else:
        matches = get_close_matches(seed_lower, book_indices.index, n=1, cutoff=0.6)
        if matches:
            matched_title = matches[0]
            print(f"Using closest match: '{matched_title}'")
            seed_idx = book_indices[matched_title]
        else:
            print(f"Seed book '{seed_title}' not found. Falling back to CF-only.")
    
    # 3. Content scores 
    if seed_idx is not None:
        if isinstance(seed_idx, pd.Series):
            seed_idx = seed_idx.iloc[0]

        seed_vector = tfidf_matrix[seed_idx]
        content_scores = cosine_similarity(seed_vector, tfidf_matrix).flatten()
        isbn_content = dict(zip(book_meta['ISBN'], content_scores))
    else:
        isbn_content = {}

    # 4. CF scores 
    u_idx = user2idx[user_id]
    all_preds = user_means[u_idx] + U_sigma[u_idx] @ Vt
    all_preds = np.clip(all_preds, 1.0, 10.0)

    rated_isbns = set(df[df['User-ID'] == user_id]['ISBN'].values)

    cand_df = book_meta.copy()
    cand_df['b_idx'] = cand_df['ISBN'].map(isbn2idx)
    cand_df = cand_df.dropna(subset=['b_idx'])
    cand_df['b_idx'] = cand_df['b_idx'].astype(int)
    cand_df = cand_df[~cand_df['ISBN'].isin(rated_isbns)]

    cf_vals = all_preds[cand_df['b_idx'].values]

    # Normalize CF scores
    cf_min, cf_max = cf_vals.min(), cf_vals.max()
    if cf_max > cf_min:
        cf_norm = (cf_vals - cf_min) / (cf_max - cf_min)
    else:
        cf_norm = np.zeros_like(cf_vals)

    # 5. Combine scores 
    cand_df = cand_df.copy()
    cand_df['cf_score_norm'] = cf_norm

    # If content exists → use it, else fallback to 0
    cand_df['content_score'] = cand_df['ISBN'].map(isbn_content).fillna(0.0)

    # Hybrid score
    cand_df['hybrid_score'] = (
        alpha * cand_df['cf_score_norm'] +
        (1 - alpha) * cand_df['content_score']
    )

    # 6. Final results
    results = (
        cand_df[['ISBN', 'Book-Title', 'Book-Author',
                 'hybrid_score', 'cf_score_norm', 'content_score']]
        .sort_values('hybrid_score', ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

    if results.empty:
        print("No recommendations found. Try another seed book.")
        return None

    results[['hybrid_score', 'cf_score_norm', 'content_score']] = \
        results[['hybrid_score', 'cf_score_norm', 'content_score']].round(4)

    return results

In [59]:
print(f"Hybrid Recommendations for User {sample_user} (seed: 'Classical Mythology'):")
hybrid_result = hybrid_recommend(sample_user, 'Decision in Normandy', n=5, alpha=0.6)
if hybrid_result is not None:
    print(hybrid_result.to_string())

Hybrid Recommendations for User 11676 (seed: 'Classical Mythology'):
Using closest match: 'delusions of grandma'
         ISBN                                                                  Book-Title         Book-Author  hybrid_score  cf_score_norm  content_score
0  0684809001                                                              The FIRST TIME                Cher        0.6786         0.7026         0.6427
1  0684863472  A Heartbreaking Work Of Staggering Genius : A Memoir Based on a True Story         Dave Eggers        0.6318         0.8152         0.3566
2  0671027360                                                         Angels &amp; Demons           Dan Brown        0.6231         1.0000         0.0578
3  0671638688                                                       It's Always Something        Gilda Radner        0.6081         0.5377         0.7136
4  0671683063                                                             Book of Virtues  William J. Bennett        

In [56]:
book_meta[book_meta['Book-Title'].str.contains('Normandy', case=False)]

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,content
1353,0743216458,"Band of Brothers : E Company, 506th Regiment, ...",Stephen E. Ambrose,2001.0,Simon &amp; Schuster,http://images.amazon.com/images/P/0743216458.0...,http://images.amazon.com/images/P/0743216458.0...,http://images.amazon.com/images/P/0743216458.0...,"band of brothers : e company, 506th regiment, ..."
8621,074322454X,"Band of Brothers : E Company, 506th Regiment, ...",Stephen E. Ambrose,2001.0,Simon &amp; Schuster,http://images.amazon.com/images/P/074322454X.0...,http://images.amazon.com/images/P/074322454X.0...,http://images.amazon.com/images/P/074322454X.0...,"band of brothers : e company, 506th regiment, ..."


In [57]:
hybrid_recommend(sample_user, 'Decision in Normandy', n=5)

Seed book 'Decision in Normandy' not found.


STEP 7: MODEL EVALUATION

In [ ]:
#7a. RMSE & MAE on test set 
# Only evaluate on test rows where both user and book are in the training set
eval_df = test_df[
    test_df['User-ID'].isin(user2idx) & test_df['ISBN'].isin(isbn2idx)
].copy()

In [62]:
eval_df['u_idx'] = eval_df['User-ID'].map(user2idx).astype(int)
eval_df['b_idx'] = eval_df['ISBN'].map(isbn2idx).astype(int)

In [63]:
# Vectorised batch prediction
pred_ratings = (
    user_means[eval_df['u_idx'].values] +
    np.einsum('ij,ij->i', U_sigma[eval_df['u_idx'].values], Vt[:, eval_df['b_idx'].values].T)
)

In [64]:
pred_ratings = np.clip(pred_ratings, 1.0, 10.0)
true_ratings = eval_df['Book-Rating'].values.astype(float)
 
rmse = np.sqrt(np.mean((pred_ratings - true_ratings) ** 2))
mae  = np.mean(np.abs(pred_ratings - true_ratings))

In [65]:
print(f"Collaborative Filtering (SVD) on Test Set:")
print(f"  RMSE : {rmse:.4f}  (target < 1.5 for 1-10 scale)")
print(f"  MAE  : {mae:.4f}  (Mean Absolute Error)")
print(f"  Evaluated on {len(eval_df):,} test pairs")

Collaborative Filtering (SVD) on Test Set:
  RMSE : 1.8520  (target < 1.5 for 1-10 scale)
  MAE  : 1.3274  (Mean Absolute Error)
  Evaluated on 27,443 test pairs


5-Fold Cross Validation

In [66]:
from sklearn.model_selection import KFold

In [67]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmses, fold_maes = [], []

In [71]:
for fold, (tr_idx, te_idx) in enumerate(kf.split(df), 1):
    tr = df.iloc[tr_idx]
    te = df.iloc[te_idx]
 
    # Build sparse matrix for this fold
    r  = tr['u_idx'].values
    c  = tr['b_idx'].values
    v  = tr['Book-Rating'].values.astype(float)
    Rf = csr_matrix((v, (r, c)), shape=(n_u, n_b))
 
    # User means
    um = np.zeros(n_u)
    for u in range(n_u):
        row_ = Rf.getrow(u)
        if row_.nnz > 0:
            um[u] = row_.data.mean()
 
    # Mean-centre
    Rc = Rf.copy().astype(float)
    for u in range(n_u):
        if Rc.getrow(u).nnz > 0:
            Rc.data[Rc.indptr[u]:Rc.indptr[u+1]] -= um[u]
 
    Uf, sf, Vtf = randomized_svd(Rc, n_components=N_FACTORS, random_state=42)
    USf = Uf * sf
 
    te_eval = te[te['User-ID'].isin(user2idx) & te['ISBN'].isin(isbn2idx)].copy()
    te_eval['u_idx'] = te_eval['User-ID'].map(user2idx).astype(int)
    te_eval['b_idx'] = te_eval['ISBN'].map(isbn2idx).astype(int)
 
    p = (
        um[te_eval['u_idx'].values] +
        np.einsum('ij,ij->i', USf[te_eval['u_idx'].values], Vtf[:, te_eval['b_idx'].values].T)
    )
    p = np.clip(p, 1.0, 10.0)
    t = te_eval['Book-Rating'].values.astype(float)
 
    fold_rmses.append(np.sqrt(np.mean((p - t) ** 2)))
    fold_maes.append(np.mean(np.abs(p - t)))
    print(f"  Fold {fold}: RMSE={fold_rmses[-1]:.4f}  MAE={fold_maes[-1]:.4f}")
 
print(f"  CV RMSE : {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")
print(f"  CV MAE  : {np.mean(fold_maes):.4f}  ± {np.std(fold_maes):.4f}")

  Fold 1: RMSE=1.8520  MAE=1.3274
  Fold 2: RMSE=1.8308  MAE=1.3164
  Fold 3: RMSE=1.8497  MAE=1.3356
  Fold 4: RMSE=1.8457  MAE=1.3226
  Fold 5: RMSE=1.8247  MAE=1.3085
  CV RMSE : 1.8406 ± 0.0108
  CV MAE  : 1.3221  ± 0.0092


In [72]:
def precision_recall_at_k(eval_frame, pred_col, true_col, uid_col, k=10, threshold=7.0):
    """
    Compute Precision@K and Recall@K averaged over all users.
    Works on a DataFrame with columns: user_id, predicted_rating, true_rating.
    """
    user_groups = eval_frame.groupby(uid_col)
    precisions, recalls = [], []
 
    for uid, grp in user_groups:
        grp_sorted = grp.sort_values(pred_col, ascending=False)
        top_k      = grp_sorted.head(k)
 
        n_rel_and_rec_k = ((top_k[true_col] >= threshold)).sum()
        n_rel           = ((grp[true_col] >= threshold)).sum()
        n_rec_k         = ((top_k[pred_col] >= threshold)).sum()
 
        prec = n_rel_and_rec_k / n_rec_k if n_rec_k  > 0 else 0.0
        rec  = n_rel_and_rec_k / n_rel   if n_rel    > 0 else 0.0
        precisions.append(prec)
        recalls.append(rec)
 
    return float(np.mean(precisions)), float(np.mean(recalls))

In [73]:
eval_df['predicted_rating'] = pred_ratings
 
prec_at_10, rec_at_10 = precision_recall_at_k(eval_df, 'predicted_rating', 'Book-Rating', 'User-ID', k=10)
prec_at_5,  rec_at_5  = precision_recall_at_k(eval_df, 'predicted_rating', 'Book-Rating', 'User-ID', k=5)

In [74]:
print(f"\nRanking Metrics (threshold = 7/10 → 'liked'):")
print(f"  Precision@5  : {prec_at_5:.4f}  ({prec_at_5*100:.1f}% of top-5 were liked)")
print(f"  Recall@5     : {rec_at_5:.4f}")
print(f"  Precision@10 : {prec_at_10:.4f}  ({prec_at_10*100:.1f}% of top-10 were liked)")
print(f"  Recall@10    : {rec_at_10:.4f}")



Ranking Metrics (threshold = 7/10 → 'liked'):
  Precision@5  : 0.6143  (61.4% of top-5 were liked)
  Recall@5     : 0.8140
  Precision@10 : 0.6156  (61.6% of top-10 were liked)
  Recall@10    : 0.8537
